# DA2-EDA-01 — t-SNE Visualization

> **Yêu cầu:** Chạy `DA2-MODEL-01` và `DA2-MODEL-04` trước

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

X_svd   = np.load('/home/jovyan/work/sv4/shared/X_svd.npy')
y_train = np.load('/home/jovyan/work/sv4/shared/y_train.npy')
y_test  = np.load('/home/jovyan/work/sv4/shared/y_test.npy')
y_all   = np.concatenate([y_train, y_test])

# Load RSI từ parquet để tô màu
df = pd.read_parquet('/home/jovyan/work/sv3/DA2-DATA-06/processed_data/bitcoin.parquet')
df = df.sort_values('timestamp').reset_index(drop=True)

print(f'X_svd: {X_svd.shape} | y_all: {y_all.shape}')

In [ ]:
print('Đang chạy t-SNE... (~30 giây)')
tsne = TSNE(n_components=2, perplexity=30, random_state=42,
            n_iter=1000, learning_rate='auto', init='pca')
X_tsne = tsne.fit_transform(X_svd)
print('t-SNE hoàn thành. Shape:', X_tsne.shape)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Buy vs Sell
for val, name, color in zip([0, 1], ['Sell', 'Buy'], ['#e74c3c', '#2ecc71']):
    mask = y_all == val
    axes[0].scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                    c=color, label=name, alpha=0.5, s=10)
axes[0].set_title('t-SNE — Tín hiệu Buy/Sell')
axes[0].set_xlabel('t-SNE Component 1')
axes[0].set_ylabel('t-SNE Component 2')
axes[0].legend(markerscale=3)

# Plot 2: màu theo RSI
sc = axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1],
                     c=df['RSI'].values, cmap='RdYlGn', alpha=0.5, s=10)
plt.colorbar(sc, ax=axes[1], label='RSI')
axes[1].set_title('t-SNE — Màu theo RSI')
axes[1].set_xlabel('t-SNE Component 1')
axes[1].set_ylabel('t-SNE Component 2')

plt.suptitle('t-SNE Visualization (sau TruncatedSVD)', fontsize=13)
plt.tight_layout()
plt.savefig('/home/jovyan/work/sv4/DA2-EDA-01/tSNE_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('Đã lưu ảnh tSNE_visualization.png')

In [ ]:
# Nhận xét
buy_tsne  = X_tsne[y_all == 1]
sell_tsne = X_tsne[y_all == 0]
print('Nhận xét t-SNE:')
print(f'  Buy  centroid : ({buy_tsne[:,0].mean():.2f}, {buy_tsne[:,1].mean():.2f})')
print(f'  Sell centroid : ({sell_tsne[:,0].mean():.2f}, {sell_tsne[:,1].mean():.2f})')
dist = np.sqrt((buy_tsne[:,0].mean()-sell_tsne[:,0].mean())**2 +
               (buy_tsne[:,1].mean()-sell_tsne[:,1].mean())**2)
print(f'  Khoảng cách centroid: {dist:.2f}')
print('  → Nếu khoảng cách lớn: Buy/Sell tách được rõ trong không gian 2D')